# Filter and reshape plasma tau tables
This notebook reads `C:\Users\okkam\Desktop\99453_prelevements_biomarqueurs.tsv.xlsx`, keeps only participants whose prefix appears in `participant_list_orig.csv`, and writes wide tables for plasma `ptau217`, `ptau181`, and `ptau231` with timepoints `t00`, `t02`, `t06`, and `t08`.

In [16]:
import pandas as pd

# 1. Define your file paths (using 'r' before the string handles the backslashes in Windows paths)
file1_path = r"C:\Users\okkam\Desktop\99453_prelevements_biomarqueurs.tsv.xlsx"
file2_path = r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\2026-05-02_glu_moca_struc_func.xlsx"

# 2. Load the Excel files into pandas DataFrames
df_biomarqueurs = pd.read_excel(file1_path)
df_participants = pd.read_excel(file2_path)

# Optional but recommended: Ensure both ID columns are of the same type (strings) 
# so they match perfectly even if one was read as text and the other as a number.
df_biomarqueurs['id_participant'] = df_biomarqueurs['id_participant'].astype(str)
df_participants['PSCID'] = df_participants['PSCID'].astype(str)

# 3. Get the unique list of your desired participant IDs
valid_ids = df_participants['PSCID'].unique()

# 4. Filter the first dataframe to KEEP only rows where 'id_participant' is in 'valid_ids'
filtered_df = df_biomarqueurs[df_biomarqueurs['id_participant'].isin(valid_ids)]

# 5. Save the filtered data to a new Excel file on your Desktop
output_path = r"C:\Users\okkam\Desktop\filtered_99453_prelevements.xlsx"
filtered_df.to_excel(output_path, index=False)

print(f"Filtering complete! Kept {len(filtered_df)} rows.")
print(f"File saved to: {output_path}")

Filtering complete! Kept 212 rows.
File saved to: C:\Users\okkam\Desktop\filtered_99453_prelevements.xlsx


In [17]:
import pandas as pd
import numpy as np

# Suppress the Future Warning you saw earlier
pd.set_option('future.no_silent_downcasting', True)

# 1. Load the filtered file
file_path = r"C:\Users\okkam\Desktop\filtered_99453_prelevements.xlsx"
df = pd.read_excel(file_path)

# 2. Define the columns we want to check
columns_to_check = [
    '99453_analyse_serum_adiponectine',
    '99453_analyse_serum_fgf21',
    '99453_analyse_serum_igfbp2',
    '99453_analyse_plasma_ptau217',
    '99453_analyse_plasma_ptau181',
    '99453_analyse_plasma_ptau231',
    '99453_analyse_csf_ab38',
    '99453_analyse_csf_ab40',
    '99453_analyse_csf_ab42',
    '99453_analyse_csf_tau_total'
]

# 3. Clean missing values (turn text into actual empty NaN values)
missing_values = ['donnÃ©e_non_disponible', 'donnée_non_disponible', 'non', 'NaN']
df = df.replace(missing_values, np.nan)

# 4. Filter participants
# how='all' means it will only drop the row if EVERY column in the subset is NaN.
# If they have even just 1 valid data point, the row is kept.
filtered_df = df.dropna(subset=columns_to_check, how='all')

# 5. Save the result to a new file
output_path = r"C:\Users\okkam\Desktop\filtered_99453_prelevements_with_data.xlsx"
filtered_df.to_excel(output_path, index=False)

# 6. Print a summary to see how many were kept vs dropped
original_count = len(df)
new_count = len(filtered_df)
print(f"Original participants: {original_count}")
print(f"Participants kept (with at least 1 data point): {new_count}")
print(f"Participants dropped (no data at all): {original_count - new_count}")
print(f"Saved to: {output_path}")

Original participants: 212
Participants kept (with at least 1 data point): 53
Participants dropped (no data at all): 159
Saved to: C:\Users\okkam\Desktop\filtered_99453_prelevements_with_data.xlsx


In [19]:
import pandas as pd

# 1. Define the file paths
file_main = r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\2026-05-02_glu_moca_struc_func.xlsx"
file_biomarqueurs = r"C:\Users\okkam\Desktop\filtered_99453_prelevements_with_data - Copy.xlsx"

# 2. Load the Excel files
df_main = pd.read_excel(file_main)
df_bio = pd.read_excel(file_biomarqueurs)

# 3. Aggressive ID Cleaning
# Convert to string -> strip invisible spaces -> remove any hidden ".0" decimals
df_main['PSCID'] = df_main['PSCID'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
df_bio['id_participant'] = df_bio['id_participant'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

# 4. Perform the Left Join
merged_df = pd.merge(df_main, df_bio, left_on='PSCID', right_on='id_participant', how='left')

# Drop the duplicate 'id_participant' column
if 'id_participant' in merged_df.columns:
    merged_df = merged_df.drop(columns=['id_participant'])

# 5. Save the final dataset
output_path = r"C:\Users\okkam\Desktop\final_article_data_with_biomarkers.xlsx"
merged_df.to_excel(output_path, index=False)

# 6. Quick check to see if it worked
# Let's count how many rows actually got a biomarker value matched 
# (picking one column to check, change '99453_analyse_adn_apoe' if it's not in your data)
matched_count = merged_df['99453_analyse_plasma_ptau217'].notna().sum()

print(f"Merge successful!")
print(f"Total rows in final file: {len(merged_df)}")
print(f"Rows that successfully matched and pulled in ptau217 data: {matched_count}")
print(f"Saved to: {output_path}")

Merge successful!
Total rows in final file: 75
Rows that successfully matched and pulled in ptau217 data: 53
Saved to: C:\Users\okkam\Desktop\final_article_data_with_biomarkers.xlsx
